[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/05-full-projects/ml-fullproj-heart.ipynb)

# Full Project: Heart Disease Risk Prediction (KNN, Cost-Sensitive Thresholding)

*AIBits Academy · Machine Learning End To End · Full Project*

297 patients, one carefully tuned algorithm, and the threshold decision that actually determines whether a screening tool is clinically useful — not another multi-model bake-off.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Fetch the lesson's dataset(s) into the working folder
import os, io, zipfile, urllib.request, urllib.parse

def fetch(url, target, member=None):   # public source; a zip member is extracted and renamed to `target`
    if os.path.exists(target):
        return
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    blob = urllib.request.urlopen(req, timeout=120).read()
    if member:
        blob = zipfile.ZipFile(io.BytesIO(blob)).read(member)
    open(target, 'wb').write(blob)
    print('downloaded', target)

fetch('https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data', 'processed.cleveland.data')

In [ ]:
# Imports used throughout this project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, mean_squared_error, mean_absolute_error, r2_score)

**Load the UCI Cleveland data.** The raw file has no header row, so we name the 14 columns ourselves; `num` is the 0-4 severity target.

In [ ]:
cols = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num']
df = pd.read_csv('processed.cleveland.data', header=None, names=cols)
print(df.shape)
df.head()

> **Business Problem**
>
> A cardiology screening clinic wants a low-cost triage tool: given a patient's vitals from a routine check-up, flag who should be prioritised for the expensive, time-consuming confirmatory test (angiography) — without sending every patient for follow-up, and without missing a genuine at-risk case. The project raised this exact tension in its closing callout ("the cost of a missed at-risk patient versus an unnecessary follow-up test would need the same Expected Value framework applied explicitly") without answering it operationally. This project is that worked answer: one well-tuned KNN model, and a full expected-cost threshold analysis.

> **Dataset**
>
> **303 patients, 13 clinical features** — age, sex, cp (chest pain type), trestbps (resting BP), chol, fbs, restecg, thalach (max heart rate achieved), exang (exercise-induced angina), oldpeak, slope, ca (number of major vessels), thal — plus a 0–4 severity target, binarised here to disease-present (1) vs. disease-absent (0). [Dataset source (UCI Cleveland) →](https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data)

## Step 1 — A Different Missing-Value Encoding: Literal ‘?’ Markers

The Diabetes project's data-quality trap was biologically impossible zeros. This dataset's quirk is different — genuinely missing readings are encoded as a literal `?` character in two columns:

In [ ]:
print(df.isin(['?']).sum()[lambda s: s>0])

Only 6 of 303 patients (2.0%) are affected, small enough that dropping those rows outright is safe and simpler than imputing — leaving **297 clean patients**:

In [ ]:
df = df[~df.isin(['?']).any(axis=1)].astype(float)
y = (df['num'] > 0).astype(int)   # binarise: any severity 1-4 -> disease present
X = df.drop(columns=['num'])
print(df.shape, y.value_counts().to_dict())

> **📊 Not an Imbalance Problem — a Different Kind of Threshold Question**
>
> 137/297 patients (46.1%) are positive — close to balanced, unlike the Diabetes (34.9%), Flight Risk (16.1%), or Fraud (0.167%) projects, where threshold-moving is usually discussed as a response to *class imbalance*. Here the classes are roughly even, so imbalance isn't the reason to move the threshold away from 0.5 — **asymmetric misclassification cost** is. That's a genuinely distinct reason, and it applies even to a perfectly balanced problem, which is exactly why this project isolates it as the sole variable under study.

## Step 2 — Scale Features, Then Tune K via Cross-Validation

KNN's distance calculation is scale-sensitive — `chol` (~150–400) would swamp `oldpeak` (~0–6) without scaling, exactly the same trap covered on the K-Means and KNN pages. A stratified 75/25 split keeps the 46.1% prevalence consistent in both sets, and **K itself is chosen by 5-fold cross-validation on the training set only** — the test set is never touched until the very end:

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
scaler = StandardScaler().fit(X_train)
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)
for k in [3,5,7,9,11,13,15,19,21]:
    knn = KNeighborsClassifier(n_neighbors=k)
    aucs = cross_val_score(knn, X_train_s, y_train, cv=cv, scoring='roc_auc')
    print(f"K={k:<3d}  CV AUC = {aucs.mean():.4f}")

**K=13** wins by cross-validated AUC (0.8966) — not the smallest or largest K tried, and not necessarily the same K a plain accuracy sweep would pick, which is why AUC (ranking quality across all thresholds) rather than accuracy-at-0.5 is the right criterion to select K when the very next step is about choosing a threshold other than 0.5.

## Step 3 — Held-Out Performance at the Default 0.5 Threshold

In [ ]:
knn = KNeighborsClassifier(n_neighbors=13).fit(X_train_s, y_train)
proba = knn.predict_proba(X_test_s)[:,1]
preds_05 = (proba >= 0.5).astype(int)
print(f"Test AUC:  {roc_auc_score(y_test, proba):.4f}")
print(f"Test accuracy @0.5: {accuracy_score(y_test, preds_05):.4f}")
print(f"Recall @0.5:        {recall_score(y_test, preds_05):.4f}")
print(confusion_matrix(y_test, preds_05))

An AUC of 0.91 is strong — the model ranks at-risk patients above healthy ones very reliably. But recall at the default 0.5 cutoff is only 70.6%: **10 of the 34 truly at-risk patients in the test set (29.4%) are told they're fine.** Whether that's acceptable depends entirely on what a missed case actually costs — a question accuracy and AUC alone never answer.

## Step 4 — What Should the Threshold Actually Be?

A missed diagnosis (false negative) sends an at-risk patient home undiagnosed. An unnecessary follow-up (false positive) costs one avoidable angiography appointment. These are not equally costly — as an illustrative clinical assumption (in a real deployment this ratio would come from actual cost/outcome data, not be guessed), this project uses **FN cost = 5× FP cost**, and sweeps every threshold from 0.05 to 0.90 to find the one that minimises total expected cost on the test set:

In [ ]:
FN_COST, FP_COST = 5, 1
results = []
for t in np.arange(0.05, 0.95, 0.05):
    preds = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    cost = fp*FP_COST + fn*FN_COST
    results.append((t, cost, tp/(tp+fn), tp/(tp+fp) if (tp+fp)>0 else 0))
best_t = min(results, key=lambda r: r[1])
print(f"Cost-optimal threshold: t={best_t[0]:.2f}  cost={best_t[1]}  recall={best_t[2]:.3f}  precision={best_t[3]:.3f}")

## Visualising the Expected-Cost Curve

The naive 0.5 default (gold marker) sits nowhere near the minimum. The true cost-minimising threshold (green marker) is considerably lower — lowering the bar for "flag this patient" is exactly what a 5× asymmetric cost calls for.

> **⚠ This Isn't Even a Recall-vs-Precision Trade-off**
>
> Moving from t=0.50 to t=0.25 cuts expected cost by **74.5%** (55 → 14) and lifts recall from 70.6% to **97.1%** — only 1 of 34 at-risk patients is now missed. The usual story here would be "better recall costs some precision or accuracy" — but accuracy is *also* higher at the cost-optimal threshold (86.5% vs. 79.7%). The naive 0.5 default isn't a reasonable trade-off point at all here; it's simply worse on every axis that matters, because 0.5 was never chosen for this problem — it's just where `predict()` defaults to when nobody thinks about the threshold explicitly.

## Key Business Takeaways

- A single well-tuned KNN (K=13, chosen by cross-validated AUC, never touching the test set) reaches AUC=0.910 on held-out patients — strong ranking quality achieved without a multi-model bake-off.
- The dataset is roughly balanced (46.1% positive) — proving that threshold-moving is not only an imbalance remedy. Here, purely asymmetric misclassification cost (a missed diagnosis vs. an unnecessary follow-up) is reason enough to move off 0.5.
- Under an explicit 5:1 cost assumption, the cost-optimal threshold (t=0.25) cuts expected cost 74.5% and improves recall from 70.6% to 97.1% — while *also* improving raw accuracy, showing the default threshold wasn't a defensible trade-off, just an unexamined default.

## Practice Questions

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · How balanced is it?

Store in `pos_rate` the fraction of the 297 clean patients with heart disease (`y == 1`).

In [ ]:
pos_rate = None   # TODO


In [ ]:
try:
    check("about 46.1%", abs(pos_rate - 137 / 297) < 1e-9)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
pos_rate = float(y.mean())

```

</details>

### Exercise 2 · Medium · Expected cost of a threshold

Write `expected_cost(t, fn_cost=5, fp_cost=1)` using `proba` and `y_test`: predict positive when `proba >= t`, then return `fp*fp_cost + fn*fn_cost`.

In [ ]:
def expected_cost(t, fn_cost=5, fp_cost=1):
    pass   # TODO


In [ ]:
try:
    tn, fp, fn, tp = confusion_matrix(y_test, (proba >= 0.5).astype(int)).ravel()
    check("cost at 0.50 = FP + 5*FN", expected_cost(0.50) == fp + 5 * fn)
    check("a lower threshold is cheaper when misses cost 5x", expected_cost(0.25) < expected_cost(0.50))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
def expected_cost(t, fn_cost=5, fp_cost=1):
    tn, fp, fn, tp = confusion_matrix(y_test, (proba >= t).astype(int)).ravel()
    return int(fp * fp_cost + fn * fn_cost)

```

</details>

### Exercise 3 · Stretch · If a miss costs ten times more

Sweep thresholds 0.05, 0.10, ..., 0.90 with `fn_cost=10`. Store the cost-minimising threshold in `best_t10`. It should be no higher than the 5:1 optimum, because misses are now even more expensive.

In [ ]:
best_t10 = None   # TODO


In [ ]:
try:
    grid5 = np.arange(0.05, 0.91, 0.05)
    best_t5 = min(grid5, key=lambda t: expected_cost(t))
    check("threshold in range", 0.05 <= best_t10 <= 0.90)
    check("not above the 5:1 optimum", best_t10 <= best_t5 + 1e-9)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
grid = np.arange(0.05, 0.91, 0.05)
best_t10 = float(min(grid, key=lambda t: expected_cost(t, fn_cost=10)))

```

The more a missed case costs relative to a false alarm, the lower the optimal threshold - the cost ratio is a business input, not a modelling choice.

</details>

---
*Back to the course: **Machine Learning End To End → Full Project: Heart Disease Risk Prediction (KNN, Cost-Sensitive Thresholding)**.*